# 7 — Free-Propeller Flight Dynamics

This notebook estimates the vertical free-flight trajectory of a 3D-printed propeller after it is released from the motor.

The intended experiment is:

1. the propeller is mounted horizontally,
2. the motor spins it up to a known launch speed,
3. the propeller is released,
4. the propeller flies upward under its own stored rotational energy,
5. aerodynamic torque slows the propeller down,
6. gravity and drag bring it back down.

The notebook only performs the flight-dynamics calculation. It does not generate geometries, run XFOIL, run QPROP, or estimate mass.

## Inputs

The notebook combines three pipeline outputs:

| File | Purpose |
|---|---|
| `./csv/prop_geometrical_params.csv` | radius, ring dimensions, blade count |
| `./csv/prop_mass_estimation.csv` | hub, blade, ring, and total propeller mass |
| `./csv/qprop_batch_results.csv` | QPROP thrust, torque, power, and aerodynamic metrics |

## Output

The final output is:

```text
./csv/flight_dynamics.csv
```

with one row per `config_id`.

## Physical model

The state vector is:

$$
\mathbf{x}(t) =
\begin{bmatrix}
h(t) \\
v(t) \\
\omega(t)
\end{bmatrix}
$$

where:

- $h$ is vertical position,
- $v$ is vertical velocity,
- $\omega$ is angular velocity.

The equations of motion are:

$$
\frac{dh}{dt} = v
$$

$$
m\frac{dv}{dt}
=
T(v,\omega)
-
mg
-
D(v)
$$

$$
I\frac{d\omega}{dt}
=
-Q(v,\omega)
$$

The propeller is assumed to keep the same aerodynamic coefficient behavior as in the QPROP simulation, but scaled with angular velocity:

$$
T(v,\omega)
=
T_{\mathrm{QPROP}}(v)
\left(
\frac{\omega}{\omega_{\mathrm{ref}}}
\right)^2
$$

$$
Q(v,\omega)
=
Q_{\mathrm{QPROP}}(v)
\left(
\frac{\omega}{\omega_{\mathrm{ref}}}
\right)^2
$$

This is a reduced-order model. It is useful for ranking designs and estimating first-order flight behavior, but it is not a substitute for a full unsteady six-degree-of-freedom simulation.

## Important modelling limitation

The most important weak point is the QPROP input velocity.

If `qprop_batch_results.csv` contains only one velocity, typically `V = 0`, the notebook assumes a static thrust curve and scales it with RPM only. This usually overestimates the flight height, because the propeller experiences axial inflow while climbing.

For better predictions, run QPROP over a velocity grid such as:

```text
V = 0, 1, 2, 3, 4, 5, 6 m/s
```

Then this notebook can interpolate the QPROP thrust and torque as functions of vertical velocity.

## 1. Imports

In [1]:
from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from tqdm.auto import tqdm

c:\Users\hecto\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Constants and configuration

All values that control the calculation are defined here. This makes the notebook easy to update if the experiment, geometry definition, or QPROP batch setup changes.

In [2]:
# -----------------------------
# Project paths
# -----------------------------

BASE_DIR = Path.cwd()
CSV_DIR = BASE_DIR / "csv"

GEOMETRY_CSV_PATH = CSV_DIR / "prop_geometrical_params.csv"
MASS_CSV_PATH = CSV_DIR / "prop_mass_estimation.csv"
QPROP_RESULTS_CSV_PATH = CSV_DIR / "qprop_batch_results.csv"

FLIGHT_DYNAMICS_CSV_PATH = CSV_DIR / "flight_dynamics.csv"


# -----------------------------
# Experiment setup
# -----------------------------

# Initial propeller speed immediately after release.
# This should match the speed just before the motor releases the propeller.
RPM_LAUNCH = 4000.0

# QPROP reference RPM used in notebook 6.
# If the qprop CSV contains an rpm column, that value is used per row.
RPM_REFERENCE_DEFAULT = 4000.0

# Initial vertical position and velocity at release.
INITIAL_HEIGHT_M = 0.0
INITIAL_VERTICAL_VELOCITY_M_S = 0.0


# -----------------------------
# Physical constants
# -----------------------------

GRAVITY_M_S2 = 9.81
AIR_DENSITY_KG_M3 = 1.225


# -----------------------------
# Geometry constants
# -----------------------------

# Hub outer radius. This is the physical radius from which the free aerodynamic blade starts.
HUB_OUTER_RADIUS_MM = 8.25

# If the root station mode used in QPROP was inner, change this to 4.5.
# The flight-dynamics inertia model still uses the physical hub radius for the hidden region.
INNER_PROFILE_RADIUS_MM = 4.5

# Effective blade root for the inertia model.
# The visible/free blade mass starts outside the hub, so the physical hub radius is the safest default.
BLADE_INERTIA_ROOT_RADIUS_MM = HUB_OUTER_RADIUS_MM


# -----------------------------
# Body drag model
# -----------------------------

# Drag coefficient for the non-powered body components.
# 1.0-1.3 is a reasonable first estimate for bluff components.
BODY_DRAG_COEFFICIENT = 1.17

# The projected frontal area of the ring in axial motion is best represented by the radial
# ring thickness, not the axial ring height. This can be changed if the CAD definition changes.
RING_FRONTAL_DIMENSION_COLUMN = "ring_thickness"

# Whether to include an approximate parasitic area for stopped/slowly rotating blades.
# The rotating blade aerodynamics are already represented by QPROP, so the default is False.
INCLUDE_BLADE_PARASITIC_AREA = False

# If blade parasitic area is enabled, this multiplier is applied to an approximate planform area.
BLADE_PARASITIC_AREA_FACTOR = 0.10


# -----------------------------
# QPROP velocity handling
# -----------------------------

# If qprop_batch_results.csv has several rows per config_id at different V values,
# use interpolation in V for T and Q at the current vertical velocity.
USE_QPROP_VELOCITY_CURVE_IF_AVAILABLE = True

# Velocity passed into the QPROP curve during ascent/descent.
# "climb_only": V_qprop = max(v, 0). This avoids using negative axial velocity in QPROP.
# "absolute"  : V_qprop = abs(v).
# "static"    : V_qprop = 0 always.
AXIAL_VELOCITY_POLICY = "climb_only"

# If the flight velocity exceeds the available QPROP velocity range, clip to the nearest
# simulated velocity. This avoids uncontrolled extrapolation.
CLIP_TO_QPROP_VELOCITY_RANGE = True


# -----------------------------
# Numerical integration
# -----------------------------

TIME_LIMIT_S = 30.0

# Small time guard so the ground-impact event does not trigger immediately at t = 0.
GROUND_EVENT_TIME_GUARD_S = 1e-6

ODE_METHOD = "RK45"
ODE_RTOL = 1e-5
ODE_ATOL = 1e-8

# Number of points used for post-processing each trajectory.
POSTPROCESS_SAMPLES = 600


# -----------------------------
# Output formatting
# -----------------------------

ROUND_OUTPUT_DECIMALS = 6

OUTPUT_COLUMNS = [
    "config_id",
    "flight_ok",
    "can_liftoff",
    "rpm_launch",
    "mass_kg",
    "inertia_kg_m2",
    "frontal_area_m2",
    "T_static_N",
    "Pshaft_ref_W",
    "Q_ref_Nm",
    "T_over_W_static",
    "h_max_m",
    "flight_time_s",
    "hover_time_s",
    "v_max_up_m_s",
    "v_impact_m_s",
    "rpm_impact",
    "spin_stop_time_s",
]

print(f"Geometry CSV      : {GEOMETRY_CSV_PATH}")
print(f"Mass CSV          : {MASS_CSV_PATH}")
print(f"QPROP CSV         : {QPROP_RESULTS_CSV_PATH}")
print(f"Output CSV        : {FLIGHT_DYNAMICS_CSV_PATH}")
print(f"Launch RPM        : {RPM_LAUNCH}")
print(f"Hub outer radius  : {HUB_OUTER_RADIUS_MM} mm")
print(f"Velocity policy   : {AXIAL_VELOCITY_POLICY}")

Geometry CSV      : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\prop_geometrical_params.csv
Mass CSV          : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\prop_mass_estimation.csv
QPROP CSV         : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\qprop_batch_results.csv
Output CSV        : c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\flight_dynamics.csv
Launch RPM        : 4000.0
Hub outer radius  : 8.25 mm
Velocity policy   : climb_only


## 3. Load and validate input data

The flight model needs geometry, mass, and QPROP performance data for the same `config_id`. The code keeps the ID as the joining key so that each flight result corresponds to the correct geometry.

In [3]:
# Load input tables

geometry_df = pd.read_csv(GEOMETRY_CSV_PATH)
mass_df = pd.read_csv(MASS_CSV_PATH)
qprop_df = pd.read_csv(QPROP_RESULTS_CSV_PATH)

# Normalize common column-name variants.
geometry_df = geometry_df.rename(
    columns={
        "blade count": "blade_count",
        "ring height": "ring_height",
        "ring thickness": "ring_thickness",
        "mid radial pos": "mid_radial_pos",
        "inner chord": "inner_chord",
        "mid chord": "mid_chord",
        "outer chord": "outer_chord",
    }
)

mass_df = mass_df.rename(
    columns={
        "m_hub_g": "m_hub_kg",
        "m_blades_g": "m_blades_kg",
        "m_ring_g": "m_outer_ring_kg",
        "m_ring_kg": "m_outer_ring_kg",
        "m_total_g": "m_total_kg",
    }
)

# If an older mass file has kg values but no config_id, align it row-by-row with geometry.
# Newer pipeline versions should already contain config_id.
if "config_id" not in mass_df.columns:
    if len(mass_df) != len(geometry_df):
        raise ValueError(
            "Mass CSV has no config_id and its row count does not match the geometry CSV. "
            "Please regenerate 2_mass_estimation.ipynb with config_id as the first column."
        )
    mass_df.insert(0, "config_id", geometry_df["config_id"].to_numpy())

# If mass columns were read from an old grams-based file, convert them.
# This is only applied when magnitudes clearly look like grams.
for col in ["m_hub_kg", "m_blades_kg", "m_outer_ring_kg", "m_total_kg"]:
    if col in mass_df.columns and mass_df[col].median() > 1.0:
        mass_df[col] = mass_df[col] / 1000.0

required_geometry_cols = [
    "config_id",
    "radius",
    "ring_height",
    "ring_thickness",
    "blade_count",
    "inner_chord",
    "mid_chord",
    "outer_chord",
]

required_mass_cols = [
    "config_id",
    "m_hub_kg",
    "m_blades_kg",
    "m_outer_ring_kg",
    "m_total_kg",
]

required_qprop_cols = [
    "config_id",
    "qprop_ok",
    "V",
    "rpm",
    "T",
    "Pshaft",
]

missing_geometry = [col for col in required_geometry_cols if col not in geometry_df.columns]
missing_mass = [col for col in required_mass_cols if col not in mass_df.columns]
missing_qprop = [col for col in required_qprop_cols if col not in qprop_df.columns]

if missing_geometry:
    raise ValueError(f"Missing geometry columns: {missing_geometry}")

if missing_mass:
    raise ValueError(f"Missing mass columns: {missing_mass}")

if missing_qprop:
    raise ValueError(f"Missing QPROP columns: {missing_qprop}")

# Keep only successful QPROP rows.
qprop_df = qprop_df.copy()
qprop_df["qprop_ok"] = qprop_df["qprop_ok"].astype(bool)
qprop_ok_df = qprop_df[qprop_df["qprop_ok"]].copy()

if qprop_ok_df.empty:
    raise ValueError("No successful QPROP rows found. Run notebook 6 before running this notebook.")

# Merge geometry and mass one row per config.
base_df = geometry_df.merge(mass_df, on="config_id", how="inner", validate="one_to_one")

if base_df.empty:
    raise ValueError("No matching config_id values between geometry and mass tables.")

print(f"Geometry rows       : {len(geometry_df)}")
print(f"Mass rows           : {len(mass_df)}")
print(f"QPROP rows          : {len(qprop_df)}")
print(f"Successful QPROP    : {len(qprop_ok_df)}")
print(f"Base merged rows    : {len(base_df)}")
print(f"Unique QPROP configs: {qprop_ok_df['config_id'].nunique()}")

Geometry rows       : 5000
Mass rows           : 5000
QPROP rows          : 5000
Successful QPROP    : 4500
Base merged rows    : 5000
Unique QPROP configs: 4500


## 4. Inertia and frontal-area model

The propeller is no longer attached to the motor, so the relevant mass is the 3D-printed propeller mass only.

The moment of inertia is approximated as the sum of:

1. a solid hub disk,
2. blade mass distributed radially from the hub radius to the tip radius,
3. outer-ring mass concentrated near the tip radius.

The blade inertia approximation is:

$$
I_{\mathrm{blades}}
=
\frac{m_{\mathrm{blades}}}{3}
\left(
R^2 + R R_{\mathrm{root}} + R_{\mathrm{root}}^2
\right)
$$

The ring inertia is:

$$
I_{\mathrm{ring}}
=
m_{\mathrm{ring}} R^2
$$

The hub inertia is:

$$
I_{\mathrm{hub}}
=
\frac{1}{2}m_{\mathrm{hub}}R_{\mathrm{hub}}^2
$$

The parasitic body drag uses:

$$
D(v)
=
\frac{1}{2}\rho C_D A_{\mathrm{front}} v|v|
$$

For vertical axial motion, the ring projected area is taken as a circumferential band:

$$
A_{\mathrm{ring}}
\approx
2\pi R t_{\mathrm{ring}}
$$

where $t_{\mathrm{ring}}$ is the radial ring thickness by default.

In [4]:
def compute_moment_of_inertia(row):
    """Estimate spin-axis moment of inertia [kg m^2]."""
    radius_m = float(row["radius"]) / 1000.0
    root_m = float(BLADE_INERTIA_ROOT_RADIUS_MM) / 1000.0
    hub_radius_m = float(HUB_OUTER_RADIUS_MM) / 1000.0

    m_hub = float(row["m_hub_kg"])
    m_blades = float(row["m_blades_kg"])
    m_ring = float(row["m_outer_ring_kg"])

    if radius_m <= root_m:
        raise ValueError(
            f"Tip radius must be larger than blade root radius. "
            f"Got R={radius_m:.5f} m, root={root_m:.5f} m"
        )

    I_hub = 0.5 * m_hub * hub_radius_m**2
    I_blades = m_blades * (radius_m**2 + radius_m * root_m + root_m**2) / 3.0
    I_ring = m_ring * radius_m**2

    return I_hub + I_blades + I_ring


def compute_frontal_area(row):
    """Estimate frontal body-drag area [m^2]."""
    radius_m = float(row["radius"]) / 1000.0
    hub_radius_m = float(HUB_OUTER_RADIUS_MM) / 1000.0

    if RING_FRONTAL_DIMENSION_COLUMN not in row.index:
        raise ValueError(
            f"RING_FRONTAL_DIMENSION_COLUMN='{RING_FRONTAL_DIMENSION_COLUMN}' "
            "is not present in the geometry table."
        )

    ring_width_m = float(row[RING_FRONTAL_DIMENSION_COLUMN]) / 1000.0

    hub_area = math.pi * hub_radius_m**2
    ring_area = 2.0 * math.pi * radius_m * ring_width_m

    blade_area = 0.0
    if INCLUDE_BLADE_PARASITIC_AREA:
        mean_chord_m = np.mean([
            float(row["inner_chord"]),
            float(row["mid_chord"]),
            float(row["outer_chord"]),
        ]) / 1000.0
        blade_span_m = max(radius_m - hub_radius_m, 0.0)
        blade_area = (
            float(row["blade_count"])
            * mean_chord_m
            * blade_span_m
            * BLADE_PARASITIC_AREA_FACTOR
        )

    return hub_area + ring_area + blade_area


base_df["inertia_kg_m2"] = base_df.apply(compute_moment_of_inertia, axis=1)
base_df["frontal_area_m2"] = base_df.apply(compute_frontal_area, axis=1)

print("Moment of inertia [kg m^2]")
print(f"  min  : {base_df['inertia_kg_m2'].min():.4e}")
print(f"  mean : {base_df['inertia_kg_m2'].mean():.4e}")
print(f"  max  : {base_df['inertia_kg_m2'].max():.4e}")

print()
print("Frontal area [cm^2]")
print(f"  min  : {1e4 * base_df['frontal_area_m2'].min():.3f}")
print(f"  mean : {1e4 * base_df['frontal_area_m2'].mean():.3f}")
print(f"  max  : {1e4 * base_df['frontal_area_m2'].max():.3f}")

Moment of inertia [kg m^2]
  min  : 5.8679e-06
  mean : 3.7533e-05
  max  : 1.2745e-04

Frontal area [cm^2]
  min  : 5.908
  mean : 12.032
  max  : 27.271


## 5. QPROP performance lookup

The flight solver needs thrust and aerodynamic torque during the trajectory.

If QPROP was run at multiple vertical velocities, the solver interpolates the QPROP thrust and torque curves as functions of velocity:

$$
T_{\mathrm{ref}} = T_{\mathrm{QPROP}}(V)
$$

$$
Q_{\mathrm{ref}} = Q_{\mathrm{QPROP}}(V)
$$

If only one QPROP row is available per propeller, the solver falls back to a static model. In that case, thrust and torque only depend on RPM:

$$
T(\omega)
=
T_0
\left(
\frac{\omega}{\omega_{\mathrm{ref}}}
\right)^2
$$

$$
Q(\omega)
=
Q_0
\left(
\frac{\omega}{\omega_{\mathrm{ref}}}
\right)^2
$$

This fallback is simple and stable, but it can overestimate peak height because it does not reduce thrust during upward climb.

In [5]:
def omega_from_rpm(rpm):
    return float(rpm) * 2.0 * math.pi / 60.0


OMEGA_LAUNCH = omega_from_rpm(RPM_LAUNCH)


def qprop_velocity_from_vertical_velocity(v_m_s):
    """Map vertical velocity to the QPROP axial velocity used for interpolation."""
    if AXIAL_VELOCITY_POLICY == "climb_only":
        return max(float(v_m_s), 0.0)
    if AXIAL_VELOCITY_POLICY == "absolute":
        return abs(float(v_m_s))
    if AXIAL_VELOCITY_POLICY == "static":
        return 0.0
    raise ValueError(f"Unknown AXIAL_VELOCITY_POLICY: {AXIAL_VELOCITY_POLICY}")


def prepare_qprop_table_for_config(config_id):
    """Return a clean QPROP table for one configuration."""
    table = qprop_ok_df[qprop_ok_df["config_id"].astype(int) == int(config_id)].copy()

    if table.empty:
        return None

    # If Q is missing or invalid, estimate it from Pshaft / omega_ref.
    if "Q" not in table.columns:
        table["Q"] = np.nan

    table["rpm"] = table["rpm"].fillna(RPM_REFERENCE_DEFAULT)
    table["omega_ref"] = table["rpm"].apply(omega_from_rpm)

    q_invalid = table["Q"].isna() | ~np.isfinite(table["Q"]) | (table["Q"] <= 0)
    table.loc[q_invalid, "Q"] = table.loc[q_invalid, "Pshaft"] / table.loc[q_invalid, "omega_ref"]

    table = table.replace([np.inf, -np.inf], np.nan)
    table = table.dropna(subset=["V", "T", "Q", "Pshaft", "rpm", "omega_ref"])

    if table.empty:
        return None

    # Average duplicate velocity rows, if any.
    table = (
        table
        .groupby("V", as_index=False)
        .agg({
            "T": "mean",
            "Q": "mean",
            "Pshaft": "mean",
            "rpm": "mean",
            "omega_ref": "mean",
        })
        .sort_values("V")
        .reset_index(drop=True)
    )

    return table


def interpolate_qprop_value(table, velocity, column):
    """Interpolate or clip a QPROP quantity in axial velocity."""
    values_v = table["V"].to_numpy(dtype=float)
    values_y = table[column].to_numpy(dtype=float)

    if len(table) == 1 or not USE_QPROP_VELOCITY_CURVE_IF_AVAILABLE:
        return float(values_y[0])

    v_query = float(velocity)

    if CLIP_TO_QPROP_VELOCITY_RANGE:
        v_query = float(np.clip(v_query, values_v.min(), values_v.max()))

    return float(np.interp(v_query, values_v, values_y))


def evaluate_propeller_forces(table, vertical_velocity_m_s, omega_rad_s):
    """Return thrust [N], aerodynamic torque [Nm], and shaft power [W]."""
    omega = max(float(omega_rad_s), 0.0)

    if omega <= 0.0:
        return 0.0, 0.0, 0.0

    v_qprop = qprop_velocity_from_vertical_velocity(vertical_velocity_m_s)

    # Use the first row as the reference angular velocity.
    omega_ref = float(table["omega_ref"].iloc[0])

    if omega_ref <= 0.0:
        raise ValueError("QPROP reference omega must be positive.")

    scale = (omega / omega_ref) ** 2

    T_ref = interpolate_qprop_value(table, v_qprop, "T")
    Q_ref = interpolate_qprop_value(table, v_qprop, "Q")

    thrust = max(T_ref * scale, 0.0)
    torque = max(Q_ref * scale, 0.0)
    power = torque * omega

    return thrust, torque, power

## 6. Flight simulation

The solver integrates the vertical motion until the propeller returns to the release height.

A configuration is considered able to lift off if:

$$
T_{\mathrm{static}} > mg
$$

where $T_{\mathrm{static}}$ is the thrust at release RPM and zero vertical velocity.

If this condition is not met, the propeller cannot leave the release point in this simplified vertical model, so the flight metrics are set to zero rather than forcing a numerical integration.

In [6]:
def ground_event(t, state):
    """Stop integration when the propeller crosses the release height while moving downward."""
    if t < GROUND_EVENT_TIME_GUARD_S:
        return 1.0
    return state[0] - INITIAL_HEIGHT_M


ground_event.terminal = True
ground_event.direction = -1


def make_equations_of_motion(row, qprop_table):
    """Create the ODE function for one configuration."""
    mass_kg = float(row["m_total_kg"])
    inertia = float(row["inertia_kg_m2"])
    frontal_area = float(row["frontal_area_m2"])

    if mass_kg <= 0:
        raise ValueError("Mass must be positive.")

    if inertia <= 0:
        raise ValueError("Moment of inertia must be positive.")

    weight = mass_kg * GRAVITY_M_S2

    def eom(t, state):
        h, v, omega = state

        thrust, torque, _power = evaluate_propeller_forces(
            qprop_table,
            vertical_velocity_m_s=v,
            omega_rad_s=omega,
        )

        body_drag = (
            0.5
            * AIR_DENSITY_KG_M3
            * BODY_DRAG_COEFFICIENT
            * frontal_area
            * v
            * abs(v)
        )

        dh_dt = v
        dv_dt = (thrust - weight - body_drag) / mass_kg
        domega_dt = -torque / inertia if omega > 0.0 else 0.0

        return [dh_dt, dv_dt, domega_dt]

    return eom


def postprocess_solution(row, qprop_table, sol):
    """Extract scalar flight metrics from the ODE solution."""
    if sol.t.size == 0:
        raise ValueError("Empty ODE solution.")

    t_end = float(sol.t[-1])
    t_grid = np.linspace(0.0, t_end, POSTPROCESS_SAMPLES)

    if sol.sol is not None:
        h, v, omega = sol.sol(t_grid)
    else:
        h = np.interp(t_grid, sol.t, sol.y[0])
        v = np.interp(t_grid, sol.t, sol.y[1])
        omega = np.interp(t_grid, sol.t, sol.y[2])

    thrust_grid = np.array([
        evaluate_propeller_forces(qprop_table, vv, ww)[0]
        for vv, ww in zip(v, omega)
    ])

    weight = float(row["m_total_kg"]) * GRAVITY_M_S2
    above_weight = thrust_grid >= weight

    if np.any(above_weight):
        hover_time_s = float(t_grid[np.where(above_weight)[0][-1]])
    else:
        hover_time_s = 0.0

    spin_positive = omega > 1e-3
    if np.any(~spin_positive):
        spin_stop_time_s = float(t_grid[np.where(~spin_positive)[0][0]])
    else:
        spin_stop_time_s = np.nan

    return {
        "h_max_m": float(np.max(h)),
        "flight_time_s": t_end,
        "hover_time_s": hover_time_s,
        "v_max_up_m_s": float(np.max(v)),
        "v_impact_m_s": float(v[-1]),
        "rpm_impact": float(max(omega[-1], 0.0) * 60.0 / (2.0 * math.pi)),
        "spin_stop_time_s": spin_stop_time_s,
    }


def make_physical_base_record(row):
    """
    Create the part of the output that only depends on geometry and mass.

    This information is available even when QPROP failed or when a configuration
    is missing from qprop_batch_results.csv. Keeping it in the output makes the
    failed rows diagnosable instead of leaving physical columns empty.
    """
    return {
        "config_id": int(row["config_id"]),
        "flight_ok": False,
        "can_liftoff": False,
        "rpm_launch": float(RPM_LAUNCH),
        "mass_kg": float(row["m_total_kg"]),
        "inertia_kg_m2": float(row["inertia_kg_m2"]),
        "frontal_area_m2": float(row["frontal_area_m2"]),
        "T_static_N": np.nan,
        "Pshaft_ref_W": np.nan,
        "Q_ref_Nm": np.nan,
        "T_over_W_static": np.nan,
        "h_max_m": np.nan,
        "flight_time_s": np.nan,
        "hover_time_s": np.nan,
        "v_max_up_m_s": np.nan,
        "v_impact_m_s": np.nan,
        "rpm_impact": np.nan,
        "spin_stop_time_s": np.nan,
    }


def simulate_single_config(row):
    """Simulate one propeller configuration and return output metrics."""
    config_id = int(row["config_id"])
    base_record = make_physical_base_record(row)

    qprop_table = prepare_qprop_table_for_config(config_id)

    # If QPROP did not produce a usable aerodynamic result, the physical values
    # are still reported, but the flight solve is marked as unavailable.
    if qprop_table is None:
        return base_record

    mass_kg = float(row["m_total_kg"])
    weight = mass_kg * GRAVITY_M_S2

    thrust_static, torque_static, power_static = evaluate_propeller_forces(
        qprop_table,
        vertical_velocity_m_s=0.0,
        omega_rad_s=OMEGA_LAUNCH,
    )

    omega_ref = float(qprop_table["omega_ref"].iloc[0])
    scale_at_launch = (OMEGA_LAUNCH / omega_ref) ** 2
    q_ref = torque_static / scale_at_launch if scale_at_launch > 0 else np.nan

    t_over_w = thrust_static / weight if weight > 0 else np.nan
    can_liftoff = bool(thrust_static > weight)

    base_record.update({
        "flight_ok": True,
        "can_liftoff": can_liftoff,
        "T_static_N": float(thrust_static),
        "Pshaft_ref_W": float(power_static),
        "Q_ref_Nm": float(q_ref),
        "T_over_W_static": float(t_over_w),
    })

    if not can_liftoff:
        base_record.update({
            "h_max_m": 0.0,
            "flight_time_s": 0.0,
            "hover_time_s": 0.0,
            "v_max_up_m_s": 0.0,
            "v_impact_m_s": 0.0,
            "rpm_impact": float(RPM_LAUNCH),
            "spin_stop_time_s": np.nan,
        })
        return base_record

    initial_state = [
        float(INITIAL_HEIGHT_M),
        float(INITIAL_VERTICAL_VELOCITY_M_S),
        float(OMEGA_LAUNCH),
    ]

    eom = make_equations_of_motion(row, qprop_table)

    try:
        sol = solve_ivp(
            eom,
            t_span=(0.0, TIME_LIMIT_S),
            y0=initial_state,
            events=ground_event,
            dense_output=True,
            method=ODE_METHOD,
            rtol=ODE_RTOL,
            atol=ODE_ATOL,
        )

        if not sol.success:
            base_record["flight_ok"] = False
            return base_record

        base_record.update(postprocess_solution(row, qprop_table, sol))
        return base_record

    except Exception as exc:
        warnings.warn(f"Flight simulation failed for config_id={config_id}: {exc}")
        base_record["flight_ok"] = False
        return base_record



## 7. Batch flight simulation

This cell runs the vertical trajectory model for every configuration that has matching geometry, mass, and QPROP data.

In [7]:
records = []

for _, row in tqdm(base_df.iterrows(), total=len(base_df), desc="Flight simulations"):
    records.append(simulate_single_config(row))

flight_dynamics_df = pd.DataFrame(records)

# Ensure stable order and schema.
flight_dynamics_df = flight_dynamics_df.sort_values("config_id").reset_index(drop=True)

for col in OUTPUT_COLUMNS:
    if col not in flight_dynamics_df.columns:
        flight_dynamics_df[col] = np.nan

flight_dynamics_df = flight_dynamics_df[OUTPUT_COLUMNS]

# Round numeric columns for compact CSV output.
numeric_cols = flight_dynamics_df.select_dtypes(include=[np.number]).columns
flight_dynamics_df[numeric_cols] = flight_dynamics_df[numeric_cols].round(ROUND_OUTPUT_DECIMALS)

print(f"Simulated rows : {len(flight_dynamics_df)}")
print(f"Successful     : {int(flight_dynamics_df['flight_ok'].sum())}")
print(f"Can liftoff    : {int(flight_dynamics_df['can_liftoff'].sum())}")

flight_dynamics_df.head()

Flight simulations: 100%|██████████| 5000/5000 [02:10<00:00, 38.38it/s]

Simulated rows : 5000
Successful     : 4500
Can liftoff    : 2503


,config_id,flight_ok,can_liftoff,rpm_launch,mass_kg,inertia_kg_m2,frontal_area_m2,T_static_N,Pshaft_ref_W,Q_ref_Nm,T_over_W_static,h_max_m,flight_time_s,hover_time_s,v_max_up_m_s,v_impact_m_s,rpm_impact,spin_stop_time_s
0,0,True,True,4000.0,0.010737,0.000024,0.001056,0.2571,1.091180,0.002605,2.440980,19.220549,7.373207,2.203346,7.526460,-9.284935,1388.073307,NaN
1,1,True,True,4000.0,0.004901,0.000008,0.000591,0.1757,0.813882,0.001943,3.654536,16.300632,5.853053,1.563420,8.709653,-9.100018,908.114468,NaN
2,2,True,True,4000.0,0.025220,0.000108,0.002224,0.8104,4.737522,0.011310,3.275580,37.436255,10.466123,3.214969,11.817087,-10.522479,1103.687335,NaN
3,3,True,True,4000.0,0.006569,0.000016,0.000629,0.3085,1.442619,0.003444,4.786902,31.757538,8.381763,2.308833,13.412386,-10.647001,753.173453,NaN
4,4,True,True,4000.0,0.015513,0.000038,0.001056,0.1543,0.455321,0.001087,1.013896,0.000945,0.307488,0.101640,0.006926,-0.020636,3918.186865,NaN


## 8. Export results

In [8]:
CSV_DIR.mkdir(parents=True, exist_ok=True)
flight_dynamics_df.to_csv(FLIGHT_DYNAMICS_CSV_PATH, index=False)

print(f"Saved flight-dynamics results to: {FLIGHT_DYNAMICS_CSV_PATH}")
print(f"Rows                              : {len(flight_dynamics_df)}")
print(f"Columns                           : {list(flight_dynamics_df.columns)}")

Saved flight-dynamics results to: c:\Users\hecto\Desktop\bachelor-thesis\IDEAL-Propeller-Dataset\csv\flight_dynamics.csv
Rows                              : 5000
Columns                           : ['config_id', 'flight_ok', 'can_liftoff', 'rpm_launch', 'mass_kg', 'inertia_kg_m2', 'frontal_area_m2', 'T_static_N', 'Pshaft_ref_W', 'Q_ref_Nm', 'T_over_W_static', 'h_max_m', 'flight_time_s', 'hover_time_s', 'v_max_up_m_s', 'v_impact_m_s', 'rpm_impact', 'spin_stop_time_s']


## 9. Final checks

These checks verify that the output is usable by the next stage of the pipeline.

They do not prove that the physical model is perfect. The main physical uncertainty remains the QPROP velocity dependence. For final experimental comparison, the preferred setup is to run QPROP at several vertical velocities and let this notebook interpolate the thrust and torque curves.

In [9]:
all_ok = True


def check(condition, message):
    global all_ok
    status = "OK" if bool(condition) else "FAIL"
    print(f"[{status:4s}] {message}")
    if not condition:
        all_ok = False


check(FLIGHT_DYNAMICS_CSV_PATH.exists(), "output CSV exists")
check(list(flight_dynamics_df.columns) == OUTPUT_COLUMNS, "output columns match expected schema")
check(flight_dynamics_df["config_id"].is_unique, "config_id values are unique")
check(flight_dynamics_df["config_id"].is_monotonic_increasing, "config_id values are sorted")
check(len(flight_dynamics_df) == len(base_df), "one output row per merged geometry/mass row")
check(flight_dynamics_df["mass_kg"].notna().all() and flight_dynamics_df["mass_kg"].gt(0).all(), "all rows have positive mass")
check(flight_dynamics_df["inertia_kg_m2"].notna().all() and flight_dynamics_df["inertia_kg_m2"].gt(0).all(), "all rows have positive inertia")
check(flight_dynamics_df["frontal_area_m2"].notna().all() and flight_dynamics_df["frontal_area_m2"].gt(0).all(), "all rows have positive frontal area")

# Static thrust-to-weight is only available when QPROP produced a usable aerodynamic result.
qprop_available = flight_dynamics_df["T_static_N"].notna()
check(
    flight_dynamics_df.loc[qprop_available, "T_over_W_static"].notna().all(),
    "all QPROP-available rows have static thrust-to-weight values",
)

successful = flight_dynamics_df[flight_dynamics_df["flight_ok"]]
if not successful.empty:
    check(successful["h_max_m"].ge(0).all(), "successful rows have non-negative peak height")
    check(successful["flight_time_s"].ge(0).all(), "successful rows have non-negative flight time")
    check(successful["v_max_up_m_s"].ge(0).all(), "successful rows have non-negative max upward velocity")

readback_df = pd.read_csv(FLIGHT_DYNAMICS_CSV_PATH)
check(readback_df.shape == flight_dynamics_df.shape, "exported CSV can be read back with the same shape")

n_liftoff = int(flight_dynamics_df["can_liftoff"].sum())
n_ok = int(flight_dynamics_df["flight_ok"].sum())
n_missing_qprop = int((~qprop_available).sum())

print()
print("Flight-dynamics summary")
print("-" * 72)
print(f"Total configurations : {len(flight_dynamics_df)}")
print(f"Successful solves    : {n_ok}")
print(f"Can liftoff          : {n_liftoff}")
print(f"Missing QPROP rows   : {n_missing_qprop}")

if n_liftoff > 0:
    liftoff_df = flight_dynamics_df[flight_dynamics_df["can_liftoff"]]
    print()
    print("Liftoff configurations")
    print("-" * 72)
    print(f"Peak height [m]      : min={liftoff_df['h_max_m'].min():.3f}, mean={liftoff_df['h_max_m'].mean():.3f}, max={liftoff_df['h_max_m'].max():.3f}")
    print(f"Flight time [s]      : min={liftoff_df['flight_time_s'].min():.3f}, mean={liftoff_df['flight_time_s'].mean():.3f}, max={liftoff_df['flight_time_s'].max():.3f}")
    print(f"Max upward v [m/s]   : min={liftoff_df['v_max_up_m_s'].min():.3f}, mean={liftoff_df['v_max_up_m_s'].mean():.3f}, max={liftoff_df['v_max_up_m_s'].max():.3f}")

if n_missing_qprop > 0:
    print()
    print("QPROP COVERAGE WARNING")
    print("-" * 72)
    print("Some configurations do not have usable QPROP rows.")
    print("Their mass, inertia and frontal area are still exported, but aerodynamic and flight metrics are left as NaN.")
    print("Regenerate notebook 6 or inspect the failed QPROP input files if you need complete coverage.")

# Informative warning about QPROP velocity coverage.
qprop_velocity_counts = qprop_ok_df.groupby("config_id")["V"].nunique()
if qprop_velocity_counts.max() <= 1:
    print()
    print("MODEL WARNING")
    print("-" * 72)
    print("QPROP appears to contain only one velocity per configuration.")
    print("The flight model therefore uses a static thrust/torque curve scaled with RPM.")
    print("For better height and flight-time estimates, run QPROP over a vertical velocity grid.")

print()
print("=" * 72)
print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED")
print("=" * 72)



[OK  ] output CSV exists
[OK  ] output columns match expected schema
[OK  ] config_id values are unique
[OK  ] config_id values are sorted
[OK  ] one output row per merged geometry/mass row
[OK  ] all rows have positive mass
[OK  ] all rows have positive inertia
[OK  ] all rows have positive frontal area
[OK  ] all QPROP-available rows have static thrust-to-weight values
[OK  ] successful rows have non-negative peak height
[OK  ] successful rows have non-negative flight time
[OK  ] successful rows have non-negative max upward velocity
[OK  ] exported CSV can be read back with the same shape

Flight-dynamics summary
------------------------------------------------------------------------
Total configurations : 5000
Successful solves    : 4500
Can liftoff          : 2503
Missing QPROP rows   : 500

Liftoff configurations
------------------------------------------------------------------------
Peak height [m]      : min=0.000, mean=17.587, max=63.633
Flight time [s]      : min=0.007, mean